# Velutina — Entrenar YOLO real con 0 MB en local (todo en Colab gratis)

Pulsa **Entorno de ejecucion > Ejecutar todo**. No necesitas claves ni cuentas.
Todo ocurre en Colab: tu disco local solo recibira al final el `best.pt` (~5MB).
Cada celda de codigo es idempotente (se puede re-ejecutar) e imprime su progreso.


In [ ]:
!nvidia-smi
!pip install ultralytics -q
import torch
HAS_GPU = torch.cuda.is_available()
print('GPU disponible:', HAS_GPU)
if HAS_GPU:
    print('Nombre GPU:', torch.cuda.get_device_name(0))
else:
    print('Sin GPU: se usara CPU con epochs=3 (modo prueba).')
EPOCHS = 25 if HAS_GPU else 3
DEVICE = 0 if HAS_GPU else 'cpu'
print(f'Config: EPOCHS={EPOCHS} DEVICE={DEVICE}')


In [ ]:
# b) Descarga GBIF sin cuentas — idempotente, con progreso
import csv, os, time, urllib.parse
import requests

SPECIES = {
    'velutina': ('Vespa velutina', 150),
    'abeja': ('Apis mellifera', 150),
    'crabro': ('Vespa crabro', 80),
}
BASE = 'https://api.gbif.org/v1/occurrence/search?scientificName=<sp>&mediaType=StillImage&limit=300&offset='
ATTR = 'ATTRIBUTION.csv'
print('Base GBIF:', BASE)
print('gbif api publica, sin claves')

done = set()
if os.path.exists(ATTR):
    import csv as _csv
    with open(ATTR, encoding='utf-8') as f:
        for row in _csv.DictReader(f):
            done.add(row.get('file', ''))
    print(f'ATTRIBUTION existente: {len(done)} ficheros ya descargados, se reutilizan.')

rows = []
if os.path.exists(ATTR):
    with open(ATTR, encoding='utf-8') as f:
        rows = list(csv.DictReader(f))

for clase, (sp, target) in SPECIES.items():
    outdir = os.path.join('data', clase)
    os.makedirs(outdir, exist_ok=True)
    have = len([r for r in rows if r.get('clase') == clase])
    print(f'\n[{clase}] {sp}: objetivo={target}, ya tenemos={have}')
    offset, got = 0, have
    while got < target:
        q = urllib.parse.quote(sp)
        url = f'https://api.gbif.org/v1/occurrence/search?scientificName={q}&mediaType=StillImage&limit=300&offset={offset}'
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        res = r.json().get('results', [])
        time.sleep(0.3)
        print(f'  offset={offset} resultados={len(res)} got={got}/{target}')
        if not res:
            break
        for occ in res:
            if got >= target:
                break
            for m in occ.get('media', []) or []:
                if got >= target:
                    break
                lic = (m.get('license') or '')
                if 'creativecommons.org/licenses/by' not in lic.lower():
                    continue
                img = m.get('identifier')
                if not img or not img.lower().startswith('http'):
                    continue
                fname = f'{clase}_{got:04d}.jpg'
                fpath = os.path.join(outdir, fname)
                if fpath in done or os.path.exists(fpath):
                    continue
                try:
                    im = requests.get(img, timeout=30)
                    im.raise_for_status()
                    with open(fpath, 'wb') as f:
                        f.write(im.content)
                except Exception as e:
                    print(f'    fallo imagen {img[:80]}: {e}')
                    continue
                rows.append({'file': fpath, 'clase': clase, 'scientificName': sp, 'gbifKey': occ.get('key', ''), 'imageUrl': img, 'license': lic, 'rightsHolder': occ.get('rightsHolder', ''), 'creator': m.get('creator', '')})
                done.add(fpath)
                got += 1
                if got % 20 == 0:
                    print(f'    ... {got}/{target}')
        offset += 300
        if offset > 3000:
            break
    print(f'[{clase}] listo: {got}/{target} en {outdir}')

with open(ATTR, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['file', 'clase', 'scientificName', 'gbifKey', 'imageUrl', 'license', 'rightsHolder', 'creator'])
    w.writeheader()
    w.writerows(rows)
print(f'\nATTRIBUTION guardado: {ATTR} con {len(rows)} filas')


In [ ]:
# c) Split train/val 80/20 estratificado, semilla 42, MOVIENDO ficheros
import os, random, shutil
random.seed(42)
CLASES = ['velutina', 'abeja', 'crabro']
print('Split 80/20 estratificado, seed=42')
# Idempotencia: si ds/ ya tiene ficheros, los devolvemos a data/ y re-hacemos el split
for c in CLASES:
    for split in ['train', 'val']:
        d = os.path.join('ds', split, c)
        if os.path.isdir(d):
            for f in os.listdir(d):
                src = os.path.join(d, f)
                dst = os.path.join('data', c, f)
                if os.path.isfile(src):
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.move(src, dst)
    print(f'  [{c}] ds/ vaciado (si existia), todo de vuelta en data/{c}/')
for c in CLASES:
    src = os.path.join('data', c)
    files = sorted([f for f in os.listdir(src) if os.path.isfile(os.path.join(src, f))]) if os.path.isdir(src) else []
    rnd = random.Random(42)
    rnd.shuffle(files)
    n = len(files)
    n_train = int(n * 0.8)
    train_f, val_f = files[:n_train], files[n_train:]
    for split, names in [('train', train_f), ('val', val_f)]:
        d = os.path.join('ds', split, c)
        os.makedirs(d, exist_ok=True)
        for f in names:
            shutil.move(os.path.join(src, f), os.path.join(d, f))
    print(f'  [{c}] total={n} train={len(train_f)} val={len(val_f)} -> ds/train/{c}/ ds/val/{c}/')
print('Split listo. Formato clasificacion por carpetas.')


## Por que clasificacion y no deteccion?

GBIF da fotos con una etiqueta por foto ("esta foto es de *Vespa velutina*"), pero **no trae cajas** (bounding boxes: donde esta el bicho dentro de la foto).
Sin cajas no se puede entrenar un detector (`yolov8n.pt`) que localiza.
Por eso entrenamos un clasificador (`yolov8n-cls.pt`): dada una foto recortada/centrada dice *velutina / abeja / crabro*.
Sirve para validar el pipeline con datos reales, pero **no localiza** bichos en vuelo: ese es el siguiente paso (Kaggle/Roboflow con cajas).


In [ ]:
# d) Entrenar YOLO clasificador — idempotente
import torch
from ultralytics import YOLO
HAS_GPU = torch.cuda.is_available()
print('ultralytics YOLO clasificacion')
print('GPU:', HAS_GPU)
# YOLO("yolov8n-cls.pt") pesos base de clasificacion
model = YOLO('yolov8n-cls.pt')
if HAS_GPU:
    print('Entrenando con GPU: epochs=25 imgsz=224 batch=64 device=0')
    model.train(data='ds', epochs=25, imgsz=224, batch=64, device=0)
else:
    print('Sin GPU: modo prueba epochs=3 device=cpu')
    model.train(data='ds', epochs=3, imgsz=224, batch=64, device='cpu')
print('Entreno listo. Pesos en runs/classify/train/weights/best.pt')


In [ ]:
# e) Evaluar: model.val() + confusion + top1 por clase + informe
import glob, os
from ultralytics import YOLO
model = YOLO('runs/classify/train/weights/best.pt' if os.path.exists('runs/classify/train/weights/best.pt') else 'yolov8n-cls.pt')
print('Validando con model.val() ...')
metrics = model.val()
print('top1:', getattr(metrics, 'top1', 'ver runs/classify/val'))
print('top5:', getattr(metrics, 'top5', 'ver runs/classify/val'))
cm = glob.glob('runs/classify/val*/confusion_matrix.png') + glob.glob('runs/classify/train/confusion_matrix.png')
print('Matriz confusion:', cm if cm else 'revisa carpeta runs/classify/')
# Top1 por clase + recall velutina + FP abeja/crabro (prediccion manual sobre ds/val)
import os
clases = ['velutina', 'abeja', 'crabro']
stats = {}
for c in clases:
    d = os.path.join('ds', 'val', c)
    files = [os.path.join(d, f) for f in sorted(os.listdir(d))] if os.path.isdir(d) else []
    ok = 0
    for fp in files:
        pred = model.predict(fp, verbose=False)[0]
        name = pred.names[pred.probs.top1]
        if name == c:
            ok += 1
        else:
            stats.setdefault(f'FP_predicho_{name}_siendo_{c}', 0)
            stats[f'FP_predicho_{name}_siendo_{c}'] += 1
    acc = ok / max(1, len(files))
    stats[f'top1_{c}'] = round(acc, 4)
    stats[f'n_{c}'] = len(files)
    print(f'  top1 {c}: {acc:.3f} ({ok}/{len(files)})')
recall_vel = stats.get('top1_velutina', 0)
fp_abeja = stats.get('FP_predicho_velutina_siendo_abeja', 0)
fp_crabro = stats.get('FP_predicho_velutina_siendo_crabro', 0)
print(f'Recall velutina: {recall_vel}')
print(f'FP: abeja->velutina={fp_abeja} crabro->velutina={fp_crabro}')
with open('resultados.txt', 'w', encoding='utf-8') as f:
    f.write(f'top1_global: {getattr(metrics, "top1", "ver runs")}\n')
    for k in sorted(stats):
        f.write(f'{k}: {stats[k]}\n')
    f.write(f'recall_velutina: {recall_vel}\n')
    f.write(f'FP_abeja_como_velutina: {fp_abeja}\n')
    f.write(f'FP_crabro_como_velutina: {fp_crabro}\n')
print('Informe guardado en resultados.txt')


In [ ]:
# f) Exportar a ONNX + mostrar tamano del best.pt
import glob, os
from ultralytics import YOLO
pt = 'runs/classify/train/weights/best.pt'
model = YOLO(pt if os.path.exists(pt) else 'yolov8n-cls.pt')
print('Exportando model.export(format="onnx") ...')
out = model.export(format='onnx')
print('ONNX:', out)
cands = glob.glob('runs/classify/train*/weights/best.pt')
best = sorted(cands)[-1] if cands else None
if best:
    mb = os.path.getsize(best) / 1e6
    print(f'best.pt: {best} ({mb:.1f} MB, ~5MB)')
else:
    print('best.pt aun no encontrado, revisa runs/classify/')
print('DESCARGA: en Colab panel Archivos -> click der en best.pt -> Descargar.')
print('Luego en tu repo copialo como jetson/ai/yolov8n_velutina.pt')


## Balance honesto

**Que demuestra:** que el pipeline funciona de punta a punta con fotos reales y licencias limpias (ATTRIBUTION.csv): descarga GBIF, split estratificado, entreno `yolov8n-cls`, metricas y export ONNX. Un YOLO de verdad entrenado por ti, gratis.

**Limites:**
- Fotos centradas de GBIF (museo/aficionado) != vuelo real con tu camara, fondos y movimiento.
- Solo ~380 fotos (150 velutina, 150 abeja, 80 crabro): dataset pequeno, sensible a luz/pose/fondo.
- Clasificacion != deteccion+localizacion: dice *que* hay en un recorte, no *donde* hay bichos en una escena.

**Siguiente paso:** deteccion con cajas (bounding boxes) desde Kaggle/Roboflow y entrenar `yolov8n.pt` detector para localizar en vuelo.
